# LangChain 从入门到精通

## 章节结构

- **1. 简介**
- **2. 基本模块**
  - PromptTemplate
  - Chains
- **3. 数据存储与检索**
  - Document Loaders
  - Text Splitters
  - Vector Stores
- **4. 结合外部工具**
  - Tool 和 Agent
  - 自定义 Tool 和 Agent
- **5. 高级应用**
  - Memory
  - 自定义 Chains
  - 动态调用
- **6. LangSmith 简介与应用**

---

In [ ]:
# 第一部分: LangChain 简介

# 安装 LangChain 和相关依赖
# !pip install langchain openai chromadb

from langchain.llms import OpenAI

# 创建一个简单的语言模型实例
llm = OpenAI(model_name="text-davinci-003", temperature=0.7)

# 示例: 使用 LLM 生成文本
prompt = "用一句话描述 LangChain 的作用。"
response = llm(prompt)
print("LangChain 的作用:", response)

---

In [ ]:
# 第二部分: 基本模块

## 2.1 PromptTemplate
from langchain.prompts import PromptTemplate

# 定义一个简单的 PromptTemplate
prompt_template = PromptTemplate(
    input_variables=["product"],
    template="用一句话推销这个产品: {product}。",
)

# 使用模板生成 Prompt
prompt = prompt_template.format(product="智能手机")
print(prompt)

## 2.2 Chains
from langchain.chains import LLMChain

# 创建一个 LLMChain
chain = LLMChain(llm=llm, prompt=prompt_template)

# 使用 Chain 生成结果
result = chain.run(product="智能手表")
print("生成结果:", result)

---

In [ ]:
# 第三部分: 数据存储与检索

## 3.1 Document Loaders
from langchain.document_loaders import TextLoader

# 加载一个文本文档
loader = TextLoader("example.txt")
documents = loader.load()
print("加载的文档:", documents)

## 3.2 Text Splitters
from langchain.text_splitter import CharacterTextSplitter

# 分割长文本为小段
text_splitter = CharacterTextSplitter(chunk_size=100, chunk_overlap=20)
texts = text_splitter.split_documents(documents)
print("分割后的文本:", texts)

## 3.3 Vector Stores
from langchain.vectorstores import Chroma
from langchain.embeddings import OpenAIEmbeddings

# 创建向量存储
embedding = OpenAIEmbeddings()
vectorstore = Chroma.from_documents(texts, embedding)

# 查询向量存储
query = "智能手表的优点是什么？"
results = vectorstore.similarity_search(query)
print("检索结果:", results)

---

In [ ]:
# 第四部分: 结合外部工具

## 4.1 使用 Tools 和 Agents
from langchain.agents import Tool, initialize_agent
from langchain.tools import tool

# 定义自定义工具
@tool
def calculate_square(num: str) -> str:
    """计算一个数字的平方。"""
    return str(int(num) ** 2)

# 初始化 Agent
tools = [Tool(name="SquareCalculator", func=calculate_square)]
agent = initialize_agent(tools, llm, agent="zero-shot-react-description", verbose=True)

# 使用 Agent
response = agent.run("计算 5 的平方")
print("Agent 结果:", response)

---

In [ ]:
# 第五部分: 高级应用

## 5.1 Memory
from langchain.chains import ConversationChain
from langchain.memory import ConversationBufferMemory

# 创建带有记忆功能的对话链
memory = ConversationBufferMemory()
conversation = ConversationChain(llm=llm, memory=memory, verbose=True)

# 示例对话
conversation.predict(input="你好，LangChain 是什么？")
conversation.predict(input="可以用 LangChain 做些什么？")

## 5.2 自定义 Chains
from langchain.chains import SimpleSequentialChain

# 定义两个简单的链
chain1 = LLMChain(llm=llm, prompt=PromptTemplate(input_variables=["text"], template="概括以下内容: {text}"))
chain2 = LLMChain(llm=llm, prompt=PromptTemplate(input_variables=["summary"], template="将以下内容翻译成英文: {summary}"))

# 组合链
sequential_chain = SimpleSequentialChain(chains=[chain1, chain2])
result = sequential_chain.run("LangChain 是一个帮助开发者构建由大语言模型驱动的应用的框架。")
print(result)

---

In [ ]:
# 第六部分: LangSmith 简介与应用

# LangSmith 是 LangChain 的可视化调试和监控平台，可以帮助开发者更好地了解链的运行过程和性能。
# 安装 LangSmith
# !pip install langsmith

from langchain.visualizers import LangSmithVisualizer

# 初始化 LangSmith 可视化器
visualizer = LangSmithVisualizer(api_key="your_langsmith_api_key")

# 示例: 将一个 LLMChain 与 LangSmith 集成
chain = LLMChain(llm=llm, prompt=PromptTemplate(input_variables=["product"], template="描述这个产品: {product}"))

# 使用 LangSmith 监控链的运行
response = visualizer.run_and_log(chain, inputs={"product": "智能音箱"})

# 打印结果和可视化链接
print("链的运行结果:", response)
print("可视化链接:", visualizer.get_visualization_url())

# LangSmith 的其他功能
# - 性能分析: 查看每一步的耗时和输入输出。
# - 错误调试: 快速定位运行过程中的错误。
# - 模型对比: 测试不同模型或提示的效果。

---